# ✍️ Shakespeare's Ghostwriter — Exploration Notebook

**CodSoft ML Internship · Task 5** (Humanized version)

This notebook walks through everything I did:
1. Load Tiny Shakespeare from HuggingFace
2. Build character vocab
3. Define the CharRNN model
4. Quick training demo
5. Play with generation + temperature

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import config
from src.data_loader import load_data, CharVocab
from src.model       import CharRNN
from src.utils       import get_device, set_seed

set_seed(config.RANDOM_SEED)
device = get_device(config.DEVICE)
print('Device:', device)

## 1. Load the dataset (cached after first run)

In [ ]:
train_loader, val_loader, vocab = load_data(
    dataset_name = config.HF_DATASET_NAME,
    max_chars    = config.MAX_CHARS,
    seq_length   = config.SEQ_LENGTH,
    batch_size   = config.BATCH_SIZE,
    cache_dir    = '../' + config.DATA_CACHE_DIR,
)

print(f'Vocab size : {vocab.size}')
print(f'Sample chars: {list(vocab.char2idx.keys())[:30]}')

x, y = next(iter(train_loader))
print(f'\nBatch x shape: {x.shape}  |  y shape: {y.shape}')
print(f'Sample input  : {vocab.decode(x[0, :30].tolist())!r}')
print(f'Sample target : {vocab.decode(y[0, :30].tolist())!r}')

## 2. Build the model

In [ ]:
model = CharRNN(
    vocab_size    = vocab.size,
    embedding_dim = config.EMBEDDING_DIM,
    hidden_size   = config.HIDDEN_SIZE,
    num_layers    = config.NUM_LAYERS,
    dropout       = config.DROPOUT,
    model_type    = config.MODEL_TYPE,
).to(device)

print(model)
print(f'\nTotal parameters: {model.count_parameters():,}')

## 3. Quick 3-epoch smoke test (so you don't wait forever)

In [ ]:
import torch.nn as nn
from torch.optim.lr_scheduler import ExponentialLR

optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
scheduler = ExponentialLR(optimizer, gamma=config.LR_DECAY)

DEMO_EPOCHS = 3

for epoch in range(1, DEMO_EPOCHS + 1):
    model.train()
    total_loss = 0
    hidden = None

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        if hidden is not None:
            hidden = model.detach_hidden(hidden)
        else:
            hidden = model.init_hidden(x.size(0), device)

        optimizer.zero_grad()
        logits, hidden = model(x, hidden)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config.CLIP_GRAD)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    avg = total_loss / len(train_loader)
    print(f'Epoch {epoch}/{DEMO_EPOCHS} | loss={avg:.4f} | perplexity={2**avg:.2f}')

## 4. Generate some text with different seeds

In [ ]:
from src.generate import generate

seeds = ['To be, or not to be', 'What is a man', 'O Romeo']

for seed in seeds:
    text = generate(
        model       = model,
        vocab       = vocab,
        seed_text   = seed,
        length      = 200,
        temperature = 0.8,
        top_p       = 0.9,
        device      = device,
    )
    print('─' * 50)
    print(text)
print('─' * 50)

## 5. Temperature playground (my favorite experiment)

In [ ]:
for temp in [0.3, 0.7, 1.0, 1.3]:
    text = generate(model, vocab, 'Once upon', 150, temperature=temp, device=device)
    print(f'\n--- Temperature={temp} ---')
    print(text)